In [ ]:
import decoupler
decoupler.__version__

In this notebook, I will run PROGENy on all cell groups. I will identify cells that are upregulated in TNFa and JAK-STAT signaling, since these are common IBD therapies. I will restrict the database to genes with p-value < 1e-10 to reduce the number of input genes to a reasonable value.

# Setup

## Environment

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import decoupler as dc
import numpy as np
import pandas as pd
from itertools import product # for generating new column w/ cellstate and diagnosis

thisseed = 231103

sc.settings.seed = thisseed

colFGID = '#1E88E5'
coliCD = '#55D276'
colcCD = '#FFC107'
colUC = '#D81B60'
colIBD = '#EB6D33'

colNM = '#6b6a6a'
colNOA = '#bfbfbf'
colFR = '#11d6cc'
colIR = '#ff8c00'
colNR = '#ff1e00'

colBG = '#C2C2C2'

sc.set_figure_params(dpi=100)

## Network

In [ ]:
dc.show_resources()

In [ ]:
# Retrieving via python
progeny = dc.get_resource('PROGENy')
progeny

In [ ]:
# Filter duplicates
progeny = progeny[~progeny.duplicated(('pathway', 'genesymbol'))]
progeny

In [ ]:
set(progeny['pathway'])

In [ ]:
pathways = ['JAK-STAT', 'TNFa']
pathways

In [ ]:
progeny_tnfjak = progeny[progeny.pathway.isin(pathways)]
progeny_tnfjak

In [ ]:
progeny_tnfjak = progeny_tnfjak.sort_values('genesymbol').sort_values('pathway')
progeny_tnfjak

In [ ]:
progeny_tnfjak.p_value.astype('float')

In [ ]:
# filter p_value
progeny_tnfjak = progeny_tnfjak[progeny_tnfjak.p_value.astype('float') < 1e-10]
progeny_tnfjak

In [ ]:
progeny_tnfjak.pathway.value_counts()

In [ ]:
# set value formats
progeny_tnfjak.p_value = progeny_tnfjak.p_value.astype('float')
progeny_tnfjak.weight = progeny_tnfjak.weight.astype('float')
progeny_tnfjak

In [ ]:
progeny_tnfjak.to_csv('../../predict/out/decoupler/network/progeny/progeny_tnfjak_network.csv', index=False)

# ColEpi

## Object Setup

In [ ]:
adata_colepi = sc.read_h5ad('../../predict/data/objects/annotated/predictCOL__CB_colepi_annotated_231226.h5ad')
adata_colepi

In [ ]:
adata_colepi.layers['counts'] = adata_colepi.X.copy()  # preserve raw counts
adata_colepi.raw = adata_colepi.copy()  # set raw for easy recovery after subsetting HVGs
sc.pp.normalize_total(adata_colepi, target_sum=1e4)
sc.pp.log1p(adata_colepi)
adata_colepi

In [ ]:
sc.pp.highly_variable_genes(
    adata_colepi,
    n_top_genes=4000,           # number of genes considered HVGs, mandatory for `flavor='seurat_v3'`
    subset=False,                # don't subset the AnnData object to just the HVGs
    layer='counts',             # use the layer we saved, instead of adata_qc.X
    flavor='seurat_v3',         # variance-stabilized method, I think
    batch_key='version_10x'     # calculates HVGs separately for each batch, chooses HVGs present in most batches
)

In [ ]:
adata_colepi.X

In [ ]:
# convert to csr, necessary for decoupler
# https://github.com/saezlab/dc-py/issues/85
adata_colepi.X = adata_colepi.X.tocsr()
adata_colepi.X

In [ ]:
adata_colepi

In [ ]:
# set combination for future plotting
adata_colepi.obs['celltype_diagSub'] = adata_colepi.obs['celltype'].astype('str') + '_' + adata_colepi.obs['diagSub'].astype('str')
adata_colepi.obs['cellstate_diagSub'] = adata_colepi.obs['cellstate'].astype('str') + '_' + adata_colepi.obs['diagSub'].astype('str')

## Anti-TNF Response UMAPs

In [ ]:
sc.pl.umap(adata_colepi, color='antitnf_response', palette = [colNM, colNOA, colFR, colIR, colNR])

In [ ]:
sc.pl.umap(adata_colepi[(adata_colepi.obs.diagSub == 'UC') & (adata_colepi.obs.timepoint == 'T0')], 
           color='antitnf_response', palette = [colNOA, colIR, colNR], title = 'UC Anti-TNF Response (PreTx)')

## Score Gene Sets

### Run decoupler

In [ ]:
%%time
dc.run_mlm(
    mat = adata_colepi,
    net = progeny_tnfjak,
    source='pathway',
    target='genesymbol',
    weight='weight',
    use_raw=False,
    min_n=0
)

In [ ]:
adata_colepi

In [ ]:
adata_colepi.obsm['mlm_estimate']

In [ ]:
adata_colepi.obsm['mlm_pvals']

In [ ]:
adata_colepi.obs

In [ ]:
adata_colepi.obs[pathways] = adata_colepi.obsm['mlm_estimate'][pathways]
adata_colepi.obs

### Plot Gene Sets

In [ ]:
sc.pl.umap(
    adata_colepi,
    color=['diagSub', 'antitnf_response', 'celltype', 'cellstate'] + pathways,
    frameon=False,
    ncols=2,
    wspace=0.3,
    cmap='turbo'
)

In [ ]:
sc.pl.stacked_violin(adata_colepi, groupby='cellstate', var_names=pathways, rotation=90, cmap='turbo')

In [ ]:
sc.pl.stacked_violin(adata_colepi, groupby='celltype_diagSub', var_names=pathways, rotation=90, cmap = 'turbo')

In [ ]:
sc.pl.stacked_violin(adata_colepi, groupby='cellstate_diagSub', var_names=pathways, rotation=90, cmap = 'turbo')

## Save

In [ ]:
adata_colepi.obsm['mlm_estimate']

In [ ]:
adata_colepi.obsm['mlm_estimate'].to_csv('../../predict/out/decoupler/progeny/mlm_tnfjak/colepi__tnfjak_scoring_mlm_estimate.csv', index_label='barcode')
adata_colepi.obsm['mlm_estimate'].to_csv('../../predict/out/decoupler/progeny/mlm_tnfjak/colepi__tnfjak_scoring_mlm_pvals.csv', index_label='barcode')

# Stroma

## Object Setup

In [ ]:
adata_stroma = sc.read_h5ad('../../predict/data/objects/annotated/predictCOL__CB_stroma_annotated_231226.h5ad')
adata_stroma

In [ ]:
adata_stroma.layers['counts'] = adata_stroma.X.copy()  # preserve raw counts
adata_stroma.raw = adata_stroma.copy()  # set raw for easy recovery after subsetting HVGs
sc.pp.normalize_total(adata_stroma, target_sum=1e4)
sc.pp.log1p(adata_stroma)
adata_stroma

In [ ]:
sc.pp.highly_variable_genes(
    adata_stroma,
    n_top_genes=4000,           # number of genes considered HVGs, mandatory for `flavor='seurat_v3'`
    subset=False,                # don't subset the AnnData object to just the HVGs
    layer='counts',             # use the layer we saved, instead of adata_qc.X
    flavor='seurat_v3',         # variance-stabilized method, I think
    batch_key='version_10x'     # calculates HVGs separately for each batch, chooses HVGs present in most batches
)

In [ ]:
adata_stroma.X

In [ ]:
# convert to csr, necessary for decoupler
# https://github.com/saezlab/dc-py/issues/85
adata_stroma.X = adata_stroma.X.tocsr()
adata_stroma.X

In [ ]:
adata_stroma

In [ ]:
# set combination for future plotting
adata_stroma.obs['celltype_diagSub'] = adata_stroma.obs['celltype'].astype('str') + '_' + adata_stroma.obs['diagSub'].astype('str')
adata_stroma.obs['cellstate_diagSub'] = adata_stroma.obs['cellstate'].astype('str') + '_' + adata_stroma.obs['diagSub'].astype('str')

## Anti-TNF Response UMAPs

In [ ]:
sc.pl.umap(adata_stroma, color='antitnf_response', palette = [colNM, colNOA, colFR, colIR, colNR])

In [ ]:
sc.pl.umap(adata_stroma[(adata_stroma.obs.diagSub == 'UC') & (adata_stroma.obs.timepoint == 'T0')], 
           color='antitnf_response', palette = [colNOA, colIR, colNR], title = 'UC Anti-TNF Response (PreTx)')

## Score Gene Sets

### Run decoupler

In [ ]:
%%time
dc.run_mlm(
    mat = adata_stroma,
    net = progeny_tnfjak,
    source='pathway',
    target='genesymbol',
    weight='weight',
    use_raw=False,
    min_n=0
)

In [ ]:
adata_stroma

In [ ]:
adata_stroma.obsm['mlm_estimate']

In [ ]:
adata_stroma.obsm['mlm_pvals']

In [ ]:
adata_stroma.obs

In [ ]:
adata_stroma.obs[pathways] = adata_stroma.obsm['mlm_estimate'][pathways]
adata_stroma.obs

### Plot Gene Sets

In [ ]:
sc.pl.umap(
    adata_stroma,
    color=['diagSub', 'antitnf_response', 'celltype', 'cellstate'] + pathways,
    frameon=False,
    ncols=2,
    wspace=0.3,
    cmap='turbo'
)

In [ ]:
sc.pl.stacked_violin(adata_stroma, groupby='cellstate', var_names=pathways, rotation=90, cmap='turbo')

In [ ]:
sc.pl.stacked_violin(adata_stroma, groupby='celltype_diagSub', var_names=pathways, rotation=90, cmap = 'turbo')

In [ ]:
sc.pl.stacked_violin(adata_stroma, groupby='cellstate_diagSub', var_names=pathways, rotation=90, cmap = 'turbo')

## Save

In [ ]:
adata_stroma.obsm['mlm_estimate']

In [ ]:
adata_stroma.obsm['mlm_estimate'].to_csv('../../predict/out/decoupler/progeny/mlm_tnfjak/stroma__tnfjak_scoring_mlm_estimate.csv', index_label='barcode')
adata_stroma.obsm['mlm_estimate'].to_csv('../../predict/out/decoupler/progeny/mlm_tnfjak/stroma__tnfjak_scoring_mlm_pvals.csv', index_label='barcode')

# Myeloid

## Object Setup

In [ ]:
adata_myeloid = sc.read_h5ad('../../predict/data/objects/annotated/predictCOL__CB_myeloid_annotated_231226.h5ad')
adata_myeloid

In [ ]:
adata_myeloid.layers['counts'] = adata_myeloid.X.copy()  # preserve raw counts
adata_myeloid.raw = adata_myeloid.copy()  # set raw for easy recovery after subsetting HVGs
sc.pp.normalize_total(adata_myeloid, target_sum=1e4)
sc.pp.log1p(adata_myeloid)
adata_myeloid

In [ ]:
sc.pp.highly_variable_genes(
    adata_myeloid,
    n_top_genes=4000,           # number of genes considered HVGs, mandatory for `flavor='seurat_v3'`
    subset=False,                # don't subset the AnnData object to just the HVGs
    layer='counts',             # use the layer we saved, instead of adata_qc.X
    flavor='seurat_v3',         # variance-stabilized method, I think
    batch_key='version_10x'     # calculates HVGs separately for each batch, chooses HVGs present in most batches
)

In [ ]:
adata_myeloid.X

In [ ]:
# convert to csr, necessary for decoupler
# https://github.com/saezlab/dc-py/issues/85
adata_myeloid.X = adata_myeloid.X.tocsr()
adata_myeloid.X

In [ ]:
adata_myeloid

In [ ]:
# set combination for future plotting
adata_myeloid.obs['celltype_diagSub'] = adata_myeloid.obs['celltype'].astype('str') + '_' + adata_myeloid.obs['diagSub'].astype('str')
adata_myeloid.obs['cellstate_diagSub'] = adata_myeloid.obs['cellstate'].astype('str') + '_' + adata_myeloid.obs['diagSub'].astype('str')

## Anti-TNF Response UMAPs

In [ ]:
sc.pl.umap(adata_myeloid, color='antitnf_response', palette = [colNM, colNOA, colFR, colIR, colNR])

In [ ]:
sc.pl.umap(adata_myeloid[(adata_myeloid.obs.diagSub == 'UC') & (adata_myeloid.obs.timepoint == 'T0')], 
           color='antitnf_response', palette = [colNOA, colIR, colNR], title = 'UC Anti-TNF Response (PreTx)')

## Score Gene Sets

### Run decoupler

In [ ]:
%%time
dc.run_mlm(
    mat = adata_myeloid,
    net = progeny_tnfjak,
    source='pathway',
    target='genesymbol',
    weight='weight',
    use_raw=False,
    min_n=0
)

In [ ]:
adata_myeloid

In [ ]:
adata_myeloid.obsm['mlm_estimate']

In [ ]:
adata_myeloid.obsm['mlm_pvals']

In [ ]:
adata_myeloid.obs

In [ ]:
adata_myeloid.obs[pathways] = adata_myeloid.obsm['mlm_estimate'][pathways]
adata_myeloid.obs

### Plot Gene Sets

In [ ]:
sc.pl.umap(
    adata_myeloid,
    color=['diagSub', 'antitnf_response', 'celltype', 'cellstate'] + pathways,
    frameon=False,
    ncols=2,
    wspace=0.3,
    cmap='turbo'
)

In [ ]:
sc.pl.stacked_violin(adata_myeloid, groupby='cellstate', var_names=pathways, rotation=90, cmap='turbo')

In [ ]:
sc.pl.stacked_violin(adata_myeloid, groupby='celltype_diagSub', var_names=pathways, rotation=90, cmap = 'turbo')

In [ ]:
sc.pl.stacked_violin(adata_myeloid, groupby='cellstate_diagSub', var_names=pathways, rotation=90, cmap = 'turbo')

## Save

In [ ]:
adata_myeloid.obsm['mlm_estimate']

In [ ]:
adata_myeloid.obsm['mlm_estimate'].to_csv('../../predict/out/decoupler/progeny/mlm_tnfjak/myeloid__tnfjak_scoring_mlm_estimate.csv', index_label='barcode')
adata_myeloid.obsm['mlm_estimate'].to_csv('../../predict/out/decoupler/progeny/mlm_tnfjak/myeloid__tnfjak_scoring_mlm_pvals.csv', index_label='barcode')

# Plasma

## Object Setup

In [ ]:
adata_plasma = sc.read_h5ad('../../predict/data/objects/annotated/predictCOL__CB_plasma_annotated_231226.h5ad')
adata_plasma

In [ ]:
adata_plasma.layers['counts'] = adata_plasma.X.copy()  # preserve raw counts
adata_plasma.raw = adata_plasma.copy()  # set raw for easy recovery after subsetting HVGs
sc.pp.normalize_total(adata_plasma, target_sum=1e4)
sc.pp.log1p(adata_plasma)
adata_plasma

In [ ]:
sc.pp.highly_variable_genes(
    adata_plasma,
    n_top_genes=4000,           # number of genes considered HVGs, mandatory for `flavor='seurat_v3'`
    subset=False,                # don't subset the AnnData object to just the HVGs
    layer='counts',             # use the layer we saved, instead of adata_qc.X
    flavor='seurat_v3',         # variance-stabilized method, I think
    batch_key='version_10x'     # calculates HVGs separately for each batch, chooses HVGs present in most batches
)

In [ ]:
adata_plasma.X

In [ ]:
# convert to csr, necessary for decoupler
# https://github.com/saezlab/dc-py/issues/85
adata_plasma.X = adata_plasma.X.tocsr()
adata_plasma.X

In [ ]:
adata_plasma

In [ ]:
# set combination for future plotting
adata_plasma.obs['celltype_diagSub'] = adata_plasma.obs['celltype'].astype('str') + '_' + adata_plasma.obs['diagSub'].astype('str')
adata_plasma.obs['cellstate_diagSub'] = adata_plasma.obs['cellstate'].astype('str') + '_' + adata_plasma.obs['diagSub'].astype('str')

## Anti-TNF Response UMAPs

In [ ]:
sc.pl.umap(adata_plasma, color='antitnf_response', palette = [colNM, colNOA, colFR, colIR, colNR])

In [ ]:
sc.pl.umap(adata_plasma[(adata_plasma.obs.diagSub == 'UC') & (adata_plasma.obs.timepoint == 'T0')], 
           color='antitnf_response', palette = [colNOA, colIR, colNR], title = 'UC Anti-TNF Response (PreTx)')

## Score Gene Sets

### Run decoupler

In [ ]:
%%time
dc.run_mlm(
    mat = adata_plasma,
    net = progeny_tnfjak,
    source='pathway',
    target='genesymbol',
    weight='weight',
    use_raw=False,
    min_n=0
)

In [ ]:
adata_plasma

In [ ]:
adata_plasma.obsm['mlm_estimate']

In [ ]:
adata_plasma.obsm['mlm_pvals']

In [ ]:
adata_plasma.obs

In [ ]:
adata_plasma.obs[pathways] = adata_plasma.obsm['mlm_estimate'][pathways]
adata_plasma.obs

### Plot Gene Sets

In [ ]:
sc.pl.umap(
    adata_plasma,
    color=['diagSub', 'antitnf_response', 'celltype', 'cellstate'] + pathways,
    frameon=False,
    ncols=2,
    wspace=0.3,
    cmap='turbo'
)

In [ ]:
sc.pl.stacked_violin(adata_plasma, groupby='cellstate', var_names=pathways, rotation=90, cmap='turbo')

In [ ]:
sc.pl.stacked_violin(adata_plasma, groupby='celltype_diagSub', var_names=pathways, rotation=90, cmap = 'turbo')

In [ ]:
sc.pl.stacked_violin(adata_plasma, groupby='cellstate_diagSub', var_names=pathways, rotation=90, cmap = 'turbo')

## Save

In [ ]:
adata_plasma.obsm['mlm_estimate']

In [ ]:
adata_plasma.obsm['mlm_estimate'].to_csv('../../predict/out/decoupler/progeny/mlm_tnfjak/plasma__tnfjak_scoring_mlm_estimate.csv', index_label='barcode')
adata_plasma.obsm['mlm_estimate'].to_csv('../../predict/out/decoupler/progeny/mlm_tnfjak/plasma__tnfjak_scoring_mlm_pvals.csv', index_label='barcode')

# B Cell

## Object Setup

In [ ]:
adata_bcell = sc.read_h5ad('../../predict/data/objects/annotated/predictCOL__CB_bcell_annotated_231226.h5ad')
adata_bcell

In [ ]:
adata_bcell.layers['counts'] = adata_bcell.X.copy()  # preserve raw counts
adata_bcell.raw = adata_bcell.copy()  # set raw for easy recovery after subsetting HVGs
sc.pp.normalize_total(adata_bcell, target_sum=1e4)
sc.pp.log1p(adata_bcell)
adata_bcell

In [ ]:
sc.pp.highly_variable_genes(
    adata_bcell,
    n_top_genes=4000,           # number of genes considered HVGs, mandatory for `flavor='seurat_v3'`
    subset=False,                # don't subset the AnnData object to just the HVGs
    layer='counts',             # use the layer we saved, instead of adata_qc.X
    flavor='seurat_v3',         # variance-stabilized method, I think
    batch_key='version_10x'     # calculates HVGs separately for each batch, chooses HVGs present in most batches
)

In [ ]:
adata_bcell.X

In [ ]:
# convert to csr, necessary for decoupler
# https://github.com/saezlab/dc-py/issues/85
adata_bcell.X = adata_bcell.X.tocsr()
adata_bcell.X

In [ ]:
adata_bcell

In [ ]:
# set combination for future plotting
adata_bcell.obs['celltype_diagSub'] = adata_bcell.obs['celltype'].astype('str') + '_' + adata_bcell.obs['diagSub'].astype('str')
adata_bcell.obs['cellstate_diagSub'] = adata_bcell.obs['cellstate'].astype('str') + '_' + adata_bcell.obs['diagSub'].astype('str')

## Anti-TNF Response UMAPs

In [ ]:
sc.pl.umap(adata_bcell, color='antitnf_response', palette = [colNM, colNOA, colFR, colIR, colNR])

In [ ]:
sc.pl.umap(adata_bcell[(adata_bcell.obs.diagSub == 'UC') & (adata_bcell.obs.timepoint == 'T0')], 
           color='antitnf_response', palette = [colNOA, colIR, colNR], title = 'UC Anti-TNF Response (PreTx)')

## Score Gene Sets

### Run decoupler

In [ ]:
%%time
dc.run_mlm(
    mat = adata_bcell,
    net = progeny_tnfjak,
    source='pathway',
    target='genesymbol',
    weight='weight',
    use_raw=False,
    min_n=0
)

In [ ]:
adata_bcell

In [ ]:
adata_bcell.obsm['mlm_estimate']

In [ ]:
adata_bcell.obsm['mlm_pvals']

In [ ]:
adata_bcell.obs

In [ ]:
adata_bcell.obs[pathways] = adata_bcell.obsm['mlm_estimate'][pathways]
adata_bcell.obs

### Plot Gene Sets

In [ ]:
sc.pl.umap(
    adata_bcell,
    color=['diagSub', 'antitnf_response', 'celltype', 'cellstate'] + pathways,
    frameon=False,
    ncols=2,
    wspace=0.3,
    cmap='turbo'
)

In [ ]:
sc.pl.stacked_violin(adata_bcell, groupby='cellstate', var_names=pathways, rotation=90, cmap='turbo')

In [ ]:
sc.pl.stacked_violin(adata_bcell, groupby='celltype_diagSub', var_names=pathways, rotation=90, cmap = 'turbo')

In [ ]:
sc.pl.stacked_violin(adata_bcell, groupby='cellstate_diagSub', var_names=pathways, rotation=90, cmap = 'turbo')

## Save

In [ ]:
adata_bcell.obsm['mlm_estimate']

In [ ]:
adata_bcell.obsm['mlm_estimate'].to_csv('../../predict/out/decoupler/progeny/mlm_tnfjak/bcell__tnfjak_scoring_mlm_estimate.csv', index_label='barcode')
adata_bcell.obsm['mlm_estimate'].to_csv('../../predict/out/decoupler/progeny/mlm_tnfjak/bcell__tnfjak_scoring_mlm_pvals.csv', index_label='barcode')

# TNKILC

## Object Setup

In [ ]:
adata_tnkilc = sc.read_h5ad('../../predict/data/objects/annotated/predictCOL__CB_tnkilc_annotated_231226.h5ad')
adata_tnkilc

In [ ]:
adata_tnkilc.layers['counts'] = adata_tnkilc.X.copy()  # preserve raw counts
adata_tnkilc.raw = adata_tnkilc.copy()  # set raw for easy recovery after subsetting HVGs
sc.pp.normalize_total(adata_tnkilc, target_sum=1e4)
sc.pp.log1p(adata_tnkilc)
adata_tnkilc

In [ ]:
sc.pp.highly_variable_genes(
    adata_tnkilc,
    n_top_genes=4000,           # number of genes considered HVGs, mandatory for `flavor='seurat_v3'`
    subset=False,                # don't subset the AnnData object to just the HVGs
    layer='counts',             # use the layer we saved, instead of adata_qc.X
    flavor='seurat_v3',         # variance-stabilized method, I think
    batch_key='version_10x'     # calculates HVGs separately for each batch, chooses HVGs present in most batches
)

In [ ]:
adata_tnkilc.X

In [ ]:
# convert to csr, necessary for decoupler
# https://github.com/saezlab/dc-py/issues/85
adata_tnkilc.X = adata_tnkilc.X.tocsr()
adata_tnkilc.X

In [ ]:
adata_tnkilc

In [ ]:
# set combination for future plotting
adata_tnkilc.obs['celltype_diagSub'] = adata_tnkilc.obs['celltype'].astype('str') + '_' + adata_tnkilc.obs['diagSub'].astype('str')
adata_tnkilc.obs['cellstate_diagSub'] = adata_tnkilc.obs['cellstate'].astype('str') + '_' + adata_tnkilc.obs['diagSub'].astype('str')

## Anti-TNF Response UMAPs

In [ ]:
sc.pl.umap(adata_tnkilc, color='antitnf_response', palette = [colNM, colNOA, colFR, colIR, colNR])

In [ ]:
sc.pl.umap(adata_tnkilc[(adata_tnkilc.obs.diagSub == 'UC') & (adata_tnkilc.obs.timepoint == 'T0')], 
           color='antitnf_response', palette = [colNOA, colIR, colNR], title = 'UC Anti-TNF Response (PreTx)')

## Score Gene Sets

### Run decoupler

In [ ]:
%%time
dc.run_mlm(
    mat = adata_tnkilc,
    net = progeny_tnfjak,
    source='pathway',
    target='genesymbol',
    weight='weight',
    use_raw=False,
    min_n=0
)

In [ ]:
adata_tnkilc

In [ ]:
adata_tnkilc.obsm['mlm_estimate']

In [ ]:
adata_tnkilc.obsm['mlm_pvals']

In [ ]:
adata_tnkilc.obs

In [ ]:
adata_tnkilc.obs[pathways] = adata_tnkilc.obsm['mlm_estimate'][pathways]
adata_tnkilc.obs

### Plot Gene Sets

In [ ]:
sc.pl.umap(
    adata_tnkilc,
    color=['diagSub', 'antitnf_response', 'celltype', 'cellstate'] + pathways,
    frameon=False,
    ncols=2,
    wspace=0.3,
    cmap='turbo'
)

In [ ]:
sc.pl.stacked_violin(adata_tnkilc, groupby='cellstate', var_names=pathways, rotation=90, cmap='turbo')

In [ ]:
sc.pl.stacked_violin(adata_tnkilc, groupby='celltype_diagSub', var_names=pathways, rotation=90, cmap = 'turbo')

In [ ]:
sc.pl.stacked_violin(adata_tnkilc, groupby='cellstate_diagSub', var_names=pathways, rotation=90, cmap = 'turbo')

## Save

In [ ]:
adata_tnkilc.obsm['mlm_estimate']

In [ ]:
adata_tnkilc.obsm['mlm_estimate'].to_csv('../../predict/out/decoupler/progeny/mlm_tnfjak/tnkilc__tnfjak_scoring_mlm_estimate.csv', index_label='barcode')
adata_tnkilc.obsm['mlm_estimate'].to_csv('../../predict/out/decoupler/progeny/mlm_tnfjak/tnkilc__tnfjak_scoring_mlm_pvals.csv', index_label='barcode')